In [2]:
from langchain_community.document_loaders import JSONLoader
from langchain.document_loaders import DirectoryLoader


In [ ]:
import json

with open("./tematik_.json", encoding="utf-8") as f:
    data = json.load(f)

docs = []

def extract_leaf_paths(node, path=None):
    if path is None:
        path = []

    if isinstance(node, dict):
        for key, value in node.items():
            extract_leaf_paths(value, path + [key])

    elif isinstance(node, list):
        if all(isinstance(v, dict) and "surah" in v and "ayat" in v for v in node):
            docs.append({
                "content": " ".join(path),
                "metadata": {
                    "root": path[0],
                    "leaf": path[-1],
                    "path": " > ".join(path)
                }
            })
        else:
            for item in node:
                extract_leaf_paths(item, path)

extract_leaf_paths(data)

with open("tematik_testing.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, indent=2, ensure_ascii=False)

print(len(docs))
print(docs[-1])


# MEMFORMAT MENJADI BENTUK YANG SESUAI UNTUK VECTORSTORE

In [ ]:
import json

with open("./tematik_.json", encoding="utf-8") as f:
    data = json.load(f)

docs = []

def extract_leaf_paths(node, path=None):
    if path is None:
        path = []

    if isinstance(node, dict):
        for key, value in node.items():
            extract_leaf_paths(value, path + [key])

    elif isinstance(node, list):
        # cek apakah list ini adalah daftar ayat (leaf)
        if all(isinstance(v, dict) and "surah" in v and "ayat" in v for v in node):
            
            root = path[0]
            leaf = path[-1]
            full_path = " > ".join(path)   # path pakai tanda >

            docs.append({
                "content": " ".join(path),   # tanpa >
                "metadata": {
                    "root": root,
                    "leaf": leaf,
                    "path": full_path        # path dengan >
                }
            })

        else:
            for item in node:
                extract_leaf_paths(item, path)

extract_leaf_paths(data)

with open("tematik_filtered.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, indent=2, ensure_ascii=False)

print(f"Total leaf ditemukan: {len(docs)}")
print(docs[-1])


In [ ]:
# check_tematik.py
import json
from collections import Counter
from pathlib import Path

JSON_PATH = Path("tematik_filtered.json")

def load_json_any(json_path: Path):
    """
    Coba baca:
    1) JSON array standar
    2) JSON Lines (satu object per baris) fallback
    """
    text = json_path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError("File kosong.")

    # Coba sebagai JSON array / object biasa
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            # kalau root dict dengan key 'data' atau semacamnya
            # silakan sesuaikan; default: bungkus jadi list 1 elemen
            data = [data]
        if not isinstance(data, list):
            raise ValueError("Root bukan list; harap berformat array of objects.")
        return data, "json"
    except json.JSONDecodeError:
        pass

    # Fallback: coba sebagai JSON Lines
    items = []
    errors = 0
    for i, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            items.append(obj)
        except json.JSONDecodeError:
            errors += 1
            print(f"[WARN] Baris {i} bukan JSON valid (abaikan).")
    if items:
        return items, "jsonl"
    raise ValueError("Gagal parse JSON. Pastikan formatnya array of objects atau JSON Lines.")

def main():
    if not JSON_PATH.exists():
        print(f"File tidak ditemukan: {JSON_PATH.absolute()}")
        return

    try:
        items, mode = load_json_any(JSON_PATH)
    except Exception as e:
        print(f"Gagal membaca file: {e}")
        return

    total = len(items)
    print(f"== Ringkasan File ==")
    print(f"- Path   : {JSON_PATH}")
    print(f"- Mode   : {mode.upper()}")
    print(f"- Total item mentah : {total}")

    # Validasi minimal untuk bisa jadi Document (content string + metadata dict)
    bad_no_content = []
    bad_content_type = []
    bad_no_metadata = []
    bad_metadata_type = []
    empty_content = []

    contents = []
    roots = []
    leaves = []

    for idx, obj in enumerate(items):
        # cek struktur object
        if not isinstance(obj, dict):
            bad_content_type.append(idx)
            continue

        content = obj.get("content", None)
        metadata = obj.get("metadata", None)

        if content is None:
            bad_no_content.append(idx)
        elif not isinstance(content, str):
            bad_content_type.append(idx)
        elif not content.strip():
            empty_content.append(idx)

        if metadata is None:
            bad_no_metadata.append(idx)
        elif not isinstance(metadata, dict):
            bad_metadata_type.append(idx)

        # kumpulkan statistik ringan
        if isinstance(content, str):
            contents.append(content)
        if isinstance(metadata, dict):
            roots.append(metadata.get("root"))
            leaves.append(metadata.get("leaf"))

    # Hitung duplikat konten
    content_counts = Counter(contents)
    dup_contents = [c for c, n in content_counts.items() if n > 1]

    valid_count = total - len(set(
        bad_no_content + bad_content_type + bad_no_metadata + bad_metadata_type
    ))

    print("\n== Estimasi Dokumen yang Akan Terbentuk ==")
    print(f"- Valid (punya content:str & metadata:dict) : {valid_count}")
    print(f"- Tidak valid (detail di bawah)             : {total - valid_count}")

    # Detail masalah
    if bad_no_content:
        print(f"  • Tanpa 'content'          : {len(bad_no_content)} item")
    if bad_content_type:
        print(f"  • 'content' bukan string   : {len(bad_content_type)} item")
    if empty_content:
        print(f"  • 'content' kosong/whitespace: {len(empty_content)} item")
    if bad_no_metadata:
        print(f"  • Tanpa 'metadata'         : {len(bad_no_metadata)} item")
    if bad_metadata_type:
        print(f"  • 'metadata' bukan object  : {len(bad_metadata_type)} item")

    if dup_contents:
        print(f"\n== Duplikat 'content' ==")
        print(f"- Jumlah konten unik duplikat: {len(dup_contents)}")
        # tampilkan beberapa contoh
        for sample in dup_contents[:5]:
            print(f"  • {sample[:120]}{'...' if len(sample)>120 else ''}  (x{content_counts[sample]})")

    # Statistik ringan metadata
    roots_counter = Counter([r for r in roots if isinstance(r, str)])
    leaves_counter = Counter([l for l in leaves if isinstance(l, str)])

    print("\n== Statistik Metadata ==")
    print(f"- Jumlah root unik  : {len(roots_counter)}")
    print(f"- Top 5 root        : {roots_counter.most_common(5)}")
    print(f"- Jumlah leaf unik  : {len(leaves_counter)}")
    print(f"- Top 5 leaf        : {[(k[:40] + ('...' if len(k)>40 else ''), v) for k, v in leaves_counter.most_common(5)]}")

    # Tampilkan sampel 3 item pertama yang valid
    print("\n== Contoh Item Valid (maks 3) ==")
    shown = 0
    for obj in items:
        if (
            isinstance(obj, dict)
            and isinstance(obj.get("content"), str)
            and isinstance(obj.get("metadata"), dict)
        ):
            print(json.dumps(obj, ensure_ascii=False)[:400])
            shown += 1
            if shown >= 3:
                break
    if shown == 0:
        print("(Tidak ada contoh valid)")

if __name__ == "__main__":
    main()


# MENGOSONGKAN VECTORSTORE (OPTIONAL)

In [ ]:
from pinecone import Pinecone
import os
from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = os.getenv('INDEX_NAME1')
index = pc.Index(index_name)

# Hapus semua vector
index.delete(delete_all=True)
print(f"Semua data di index '{index_name}' sudah dihapus.")


# MENGINGEST KE DALAM VECTORSTORE

In [ ]:
from langchain_community.document_loaders import JSONLoader
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import os
from dotenv import load_dotenv

load_dotenv()
index_name =os.getenv('INDEX_NAME1')
embedding = OpenAIEmbeddings()
from tqdm import tqdm
loader = JSONLoader(
    file_path="tematik_filtered.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=lambda r, m: r["metadata"],
    json_lines=False
)

docs = loader.load()
print(docs[0])

batch_size = 100

for i in tqdm(range(0, len(docs), batch_size)):
    batch = docs[i:i+batch_size]
    PineconeVectorStore.from_documents(
        documents=batch,
        embedding=embedding,
        index_name=index_name
    )


# CEK SIMILARITY SEARCH (OPTIONAL)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import os

# Inisialisasi embedding dan vectorstore
embeddings = OpenAIEmbeddings()
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=os.environ["INDEX_NAME1"],
    embedding=embeddings
)

# Lakukan pencarian
query = "SIAPA IBU DARI NABI ISA"
docs = vectorstore.similarity_search(query, k=5)

# Tampilkan hasil
for i, doc in enumerate(docs):
    print(f"\n--- Hasil {i+1} ---")
    print("Isi:", doc.page_content)
    print("Metadata:", doc.metadata)


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import os

# Inisialisasi embedding dan vectorstore
embeddings = OpenAIEmbeddings()
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=os.environ["INDEX_NAME1"],
    embedding=embeddings
)

# Lakukan pencarian
query = "jin diciptakan"
docs = vectorstore.similarity_search(query, k=5)

# Tampilkan hasil
for i, doc in enumerate(docs):
    print(f"\n--- Hasil {i+1} ---")
    print("Isi:", doc.page_content)
    print("Metadata:", doc.metadata)
